# FPS Benchmark

Do toc do inference cua ca 3 model: Baseline CNN, DAN, POSTER
KET QUA: FPS thuc te = so khung hinh / giay ma model co the xu ly

In [ ]:
import sys, os, time
sys.path.insert(0, os.getcwd())

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 150

TEST_DIR = 'data/DATASET/test'
EMOTIONS = ['Surprise', 'Fear', 'Disgust', 'Happiness', 'Sadness', 'Anger', 'Neutral']

print('Environment ready')

In [ ]:
# Load sample images cho benchmark
import glob

image_paths = []
for cls in range(1, 8):
    cls_dir = os.path.join(TEST_DIR, str(cls))
    if os.path.exists(cls_dir):
        imgs = sorted(os.listdir(cls_dir))[:10]
        image_paths.extend([os.path.join(cls_dir, f) for f in imgs])

print(f'Collected {len(image_paths)} test images')

# Load and cache images
test_images = []
for p in image_paths:
    test_images.append(np.array(Image.open(p).convert('RGB')))
print(f'Loaded {len(test_images)} images')

In [ ]:
def benchmark(name, predict_fn, images, n_warmup=5, n_runs=50):
    """
    Benchmark FPS cua 1 model.
    predict_fn: ham nhan (img_array) -> result
    """
    # Warmup
    print(f'{name}: Warming up {n_warmup} runs...')
    for i in range(n_warmup):
        _ = predict_fn(images[i % len(images)])

    # Benchmark
    times = []
    print(f'{name}: Running {n_runs} iterations...')
    for i in range(n_runs):
        idx = i % len(images)
        start = time.perf_counter()
        _ = predict_fn(images[idx])
        elapsed = time.perf_counter() - start
        times.append(elapsed)

    times = np.array(times)
    avg_ms = np.mean(times) * 1000
    fps = 1.0 / np.mean(times)
    p50 = np.percentile(times, 50) * 1000
    p95 = np.percentile(times, 95) * 1000
    p99 = np.percentile(times, 99) * 1000

    print(f'  Avg: {avg_ms:.1f} ms/frame  =>  FPS: {fps:.1f}')
    print(f'  P50: {p50:.1f} ms | P95: {p95:.1f} ms | P99: {p99:.1f} ms')
    print()

    return {
        'name': name,
        'avg_ms': avg_ms,
        'fps': fps,
        'p50': p50,
        'p95': p95,
        'p99': p99,
        'times': times,
    }

In [ ]:
results = []

# === Baseline CNN ===
try:
    import tensorflow as tf
    from app.models.emotion_model import EmotionPredictor
    
    pred = EmotionPredictor()
    if pred.is_loaded:
        r = benchmark('Baseline CNN (TF)', pred.predict, test_images, n_runs=100)
        results.append(r)
    else:
        print('CNN: model not loaded')
except Exception as e:
    print(f'CNN error: {e}')

In [ ]:
# === DAN ===
try:
    from app.models.dan_model import DANPredictor
    
    pred = DANPredictor()
    if pred.is_loaded:
        r = benchmark('DAN (PyTorch)', pred.predict, test_images, n_runs=50)
        results.append(r)
    else:
        print('DAN: model not loaded')
except Exception as e:
    print(f'DAN error: {e}')

In [ ]:
# === POSTER ===
try:
    from app.models.poster_model import POSTERPredictor
    
    pred = POSTERPredictor()
    if pred.is_loaded:
        r = benchmark('POSTER (PyTorch)', pred.predict, test_images, n_runs=20)
        results.append(r)
    else:
        print('POSTER: model not loaded')
except Exception as e:
    print(f'POSTER error: {e}')

In [ ]:
# === INCLUDE FACE DETECTION (end-to-end) ===
try:
    from app.services.face_service import FaceDetectionService
    from app.services.prediction_service import PredictionService
    
    def full_pipeline(model_type):
        svc = PredictionService(model_type=model_type)
        def fn(img):
            import cv2, base64
            _, buffer = cv2.imencode('.jpg', cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            b64 = base64.b64encode(buffer).decode()
            svc.predict_from_base64(b64, realtime=True)
        return fn
    
    for mt, name in [('keras', 'CNN + FaceDet'), ('dan', 'DAN + FaceDet')]:
        r = benchmark(name, full_pipeline(mt), test_images[:10], n_warmup=2, n_runs=10)
        results.append(r)
except Exception as e:
    print(f'Full pipeline error: {e}')

In [ ]:
# === Plot comparison ===
if results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    names = [r['name'] for r in results]
    fps_vals = [r['fps'] for r in results]
    ms_vals = [r['avg_ms'] for r in results]
    colors = ['#4CAF50', '#2196F3', '#9C27B0', '#FF9800', '#F44336']

    ax = axes[0]
    bars = ax.barh(names, fps_vals, color=colors[:len(names)])
    for bar, v in zip(bars, fps_vals):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{v:.1f}', va='center', fontsize=10, fontweight='bold')
    ax.set_xlabel('FPS (higher = better)')
    ax.set_title('FPS Comparison')

    ax = axes[1]
    bars = ax.barh(names, ms_vals, color=colors[:len(names)])
    for bar, v in zip(bars, ms_vals):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f'{v:.1f} ms', va='center', fontsize=10, fontweight='bold')
    ax.set_xlabel('Latency (ms) (lower = better)')
    ax.set_title('Inference Time per Frame')

    plt.suptitle('FPS Benchmark - Facial Emotion Recognition', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('fps_benchmark.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

    print('='*60)
    print('SUMMARY')
    print('='*60)
    for r in results:
        print(f'{r["name"]:25s}: {r["avg_ms"]:7.1f} ms/frame  =>  {r["fps"]:5.1f} FPS')
    print('='*60)
    print('Saved: fps_benchmark.png')
else:
    print('No results to plot')

## Ghi chu
- FPS o day la **model inference speed** (khong bao gom network delay)
- End-to-end (co face detection) se cham hon do YOLOv8 mat ~20-50ms
- Tren web app thuc te, FPS thap hon do: encode/decode base64 + network latency + rendering